# トピック分析４：事前学習済みの日本語BERTで様々なタスクを解いてみよう!

## 1. BERTとは？
2018年10月頃に登場した自然言語処理の手法です。
Bidirectional Encoder Representations from Transformers の略です。
技術に関する詳細は[論文](https://arxiv.org/pdf/1810.04805.pdf)に譲りますが、
BERTは自然言語処理の各ベンチマークで非常に高い性能を示していること、
事前学習モデルのファインチューニングによって様々なタスクに適用することができることから、
多くの研究・アプリケーションで利用されています。

huggingfaceが公開しているAPIを利用することで、簡単に学習済みモデルを操作することができます。

In [ ]:
!pip install transformers
!pip install fugashi
!pip install unidic-lite
!pip install ipadic
!pip install sentencepiece==0.1.94
!pip install sacremoses

In [ ]:
from transformers import BertTokenizer, BertJapaneseTokenizer, BertForSequenceClassification, pipeline

## 2. 事前学習済みモデルのロード
BERTは大量のデータで事前に学習する必要があります。
計算には巨大な計算資源と長い時間が必要になってしまうので、ここでは事前学習済みの日本語モデルを利用します。
有名な事前学習済みの日本語BERTモデルとして、東北大が提供している `cl-tohoku/bert-large-japanese` と、huggingfaceが提供している `bert-base-multilingual-uncase` が存在しています。
どちらもwikipediaの日本語データを利用して学習していますが、前処理や学習の方法が異なるため、同じ入力をしても異なる結果がでます。

BERTは汎用的に様々なタスクをこなすことができます。
ここでは、UNMASKのタスクを試してみましょう。
学習済みモデルと単語分割器をダウンロードします。

In [ ]:
#### 東北大のモデル
model_name = 'cl-tohoku/bert-base-japanese-whole-word-masking'
# トークナイザーの読み込み
tokenizer = BertJapaneseTokenizer.from_pretrained(model_name)

'''
#### haggingfaceのモデル
model_name = 'bert-base-multilingual-cased'
# トークナイザーの読み込み
tokenizer = BertTokenizer.from_pretrained(model_name)
'''

# パイプラインの読み込み
unmasker = pipeline('fill-mask', model=model_name, tokenizer=tokenizer)

## 3. BERTに穴埋め問題を解かせてみよう！

`unmasker()` に渡す文字列に含まれる `[MASK]` の部分を BERT が予測して埋めてくれます。
渡す文章やMASK部分を変更して遊んでみましょう！
なお、この学習済みモデルはWikipediaしか知りません。よって、Wikipediaにはあらわれないような文（例えばSNSや掲示板のような口語体）の穴埋め問題を解くことは苦手です。いろいろな文を入力して確認してみましょう。

In [ ]:
# パイプラインの実行
unmasker("東京駅から[MASK]駅に向かいます。")
#unmasker("盛岡駅から[MASK]駅に向かいます。")


## 4. BERTでいろいろな問題を解いてみよう

`pipeline` にわたす引数（上のセルでは `fill-mask` となっている）を変更することで、同じモデルをベースに様々なタスクを解かせることができます（[くわしくはこちら](https://huggingface.co/docs/transformers/main_classes/pipelines)）。
ただし、日本語モデルは非対応のものが多いので、ここからは英語のモデルを使いたいと思います。
以下のコードは[こちら](https://huggingface.co/transformers/v4.5.1/task_summary.html)のページを参考にしています。
日本語のときは、unmaskにモデルと単語分割器を指定していましたが、何も指定しないとデフォルトのモデルが選ばれます。

### 4.1 まずは穴埋め問題から
穴埋めする単語を`<mask>`に置き換えて実行してください。

In [ ]:
# パイプラインの生成
unmasker = pipeline('fill-mask', model = 'distilroberta-base')
# パイプラインの実行
unmasker("He bought a gallon <mask> milk.")

### 4.2 センチメント分析
センチメント分析とは、与えられた文がポジティブな印象か、ネガティブな印象かを予測する問題です。
"I hate you"はスコア0.99でネガティブ、"I love you"はスコア0.99でポジティブと判定されています。


In [ ]:
classifier = pipeline("sentiment-analysis", model = 'distilbert-base-uncased-finetuned-sst-2-english')
print(classifier("I hate you")[0])
print(classifier("I love you")[0])


### 4,3 質問応答タスク

このタスクでは、まずどこかに解答が書かれているような、質問の前提となる文をコンテキスト（文脈）として渡します。
以下のコードで渡しているのは、Wikipediaの英語版からとってきたもので、日本語に翻訳すると以下のような内容です。

```「UTokyo は10学部、15大学院を擁し、約3万人の学生が在籍していますが、そのうち約4,200人が外国人留学生です。特に、8割以上を占める私費留学生の数は、2010年からの10年間で1.75倍に増加しています。同大学では留学生の支援に力を入れている。キャンパスは、本郷、駒場、柏、白金、中野の5カ所。日本で最も選択性の高い名門大学と言われています。2020年現在、東京大学の卒業生、教員、研究者には、17人の内閣総理大臣、17人のノーベル賞受賞者、4人のプリツカー賞受賞者が含まれています。プリツカー賞受賞者4名、宇宙飛行士5名、フィールズ賞受賞者1名などがいます。」```

これに対し、次のような質問を与えています。

1. How many international students are there at the UTokyo? (東京大学には何名の外国人留学生がいますか？)
2. Where is the campus located? (キャンパスはどこにありますか？)
3. What kind of university is the University of Tokyo known for? (東京大学はどのような大学と知られていますか？

BERTの答えは以下の通りです。

1. 4,200 (正解！）
2. Hongo (半分正解。ほかにもあります）
3. most selective and pretigious (正解！）


In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-cased-distilled-squad')
model = AutoModelForQuestionAnswering.from_pretrained('distilbert-base-cased-distilled-squad')

# 質問の前提となる文をコンテクスト（文脈）
context = r"""UTokyo has ten faculties, 15 graduate schools and enrolls about 30,000 students, about 4,200 of whom are international students.
In particular, the number of privately funded international students, who account for more than 80%, has increased 1.75 times in the 10 years since 2010,
and the university is focusing on supporting international students. Its five campuses are in Hongō, Komaba, Kashiwa, Shirokane and Nakano.
It is considered to be the most selective and prestigious university in Japan.
As of 2020, University of Tokyo's alumni, faculty members and researchers include seventeen Prime Ministers, 17 Nobel Prize laureates,
four Pritzker Prize laureates, five astronauts, and a Fields Medalist."""

questions = [
    "How many international students are there at the UTokyo?",
    "Where is the campus located?",
    "What kind of university is the University of Tokyo known for?"
]

def answer_question(question, context, tokenizer, model):
    inputs = tokenizer(question, context, return_tensors="pt", max_length=512, truncation=True)
    outputs = model(**inputs)

    answer_start_scores = outputs.start_logits
    answer_end_scores = outputs.end_logits

    # Get the most likely beginning and end of the answer.
    answer_start = answer_start_scores.argmax()
    answer_end = answer_end_scores.argmax() + 1 # +1 to include the end token

    # Convert tokens to string
    input_ids = inputs['input_ids'].tolist()[0]
    answer_tokens = input_ids[answer_start:answer_end]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

    # You can also return scores or other details if needed
    # For simplicity, returning a dictionary similar to pipeline output format
    return {'answer': answer, 'start': answer_start.item(), 'end': answer_end.item(), 'score': (answer_start_scores[0, answer_start].item() + answer_end_scores[0, answer_end-1].item()) / 2}

# 質問
for question in questions:
    print(answer_question(question=question, context=context, tokenizer=tokenizer, model=model))


### 4.4 文生成
GPT-2と呼ばれる文生成アルゴリズムによって、"I was so surprised that the university of Tokyo was"に続く文を生成してみましょう。
この結果、私が実行したときは以下のような文が生成されました。
なお、この文はランダム値をもとに生成されるため、実行するたびに違う文が生成されます。

```I was so surprised that the university of Tokyo was able to build a university. I don't know how hard it is to build a university, though... The problem that came up is the fact that my parents don't have any money (東大が大学を作れることにびっくりしました。大学を作ることがどれほど大変なことなのか、私にはわかりませんが......。問題になったのは、親がお金を持っていないことです。)```



In [ ]:
text_generator = pipeline("text-generation", model = 'gpt2', max_length=50, pad_token_id=50256)
text_generator("I was so surprised that the university of Tokyo was")[0]['generated_text']

### 4.5 固有表現認識

固有表現認識とは、与えられた文の各単語に対し、固有名詞（人名、組織名、地名など）や日付、時間表現、数量、金額、パーセンテージなどのあらかじめ定義された固有表現分類へと分類する課題です。文の意味を解釈する際の前処理として使われます。

```The University of Tokyo is a public research university located in Bunkyō, Tokyo, Japan. The president of the university is Teruo Fujii.```

を入力文として与えたところ、```University of Tokyo```がORG (組織）、```Bunkyō, Tokyo, Japan.```がLOC（地名）、```Teruo Fujii```がPER（人名）として認識されました。


In [ ]:
ner_pipe = pipeline("ner", model = 'dbmdz/bert-large-cased-finetuned-conll03-english')
sequence = """The University of Tokyo is a public research university located in Bunkyō, Tokyo, Japan. The president of the university is Teruo Fujii."""
result = ner_pipe(sequence)
for word in result:
    print(word)

### 4.6 文要約

与えられた文を指定した長さの文に要約します。
下のコードでは、[英語版Wikipediaの東京大学のHistory](https://en.wikipedia.org/wiki/University_of_Tokyo#History)を抜粋した文を与え、30単語以上130単語以下の文に要約させています。
その結果、以下のように要約されました。

``` The university was chartered by the Meiji government in 1877 under its current name . It was renamed "the Imperial University" in 1886, and then Tokyo Imperial University in 1897 . In September 1923, an earthquake and the following fires destroyed about 750,000 volumes of the Imperial University Library . ( 帝国大学は1877年に明治政府によって設立され、現在の名称になりました。1886年には「帝国大学」、1897年には「東京帝国大学」と改称されました。1923年9月の地震とその後の火災で、帝国大学図書館の約75万冊の蔵書が焼失した。)```


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load tokenizer and model for summarization
tokenizer = AutoTokenizer.from_pretrained('sshleifer/distilbart-cnn-12-6')
model = AutoModelForSeq2SeqLM.from_pretrained('sshleifer/distilbart-cnn-12-6')

ARTICLE = """The university was chartered by the Meiji government in 1877 under its current name by amalgamating older government schools for medicine, various traditional scholars and modern learning. It was renamed "the Imperial University (帝國大學, Teikoku daigaku)" in 1886, and then Tokyo Imperial University (東京帝國大學, Tōkyō teikoku daigaku) in 1897 when the Imperial University system was created. In September 1923, an earthquake and the following fires destroyed about 750,000 volumes of the Imperial University Library.[16][17] The books lost included the Hoshino Library (星野文庫, Hoshino bunko), a collection of about 10,000 books.[17][18] The books were the former possessions of Hoshino Hisashi before becoming part of the library of the university and were mainly about Chinese philosophy and history.

In 1947 after Japan's defeat in World War II it re-assumed its original name. With the start of the new university system in 1949, Todai swallowed up the former First Higher School (today's Komaba campus) and the former Tokyo Higher School, which thenceforth assumed the duty of teaching first- and second-year undergraduates, while the faculties on Hongo main campus took care of third- and fourth-year students.

Although the university was founded during the Meiji period, it has earlier roots in the Astronomy Agency (天文方; 1684), Shoheizaka Study Office (昌平坂学問所; 1797), and the Western Books Translation Agency (蕃書和解御用; 1811).[19] These institutions were government offices established by the 徳川幕府 Tokugawa shogunate (1603–1867), and played an important role in the importation and translation of books from Europe.

According to The Japan Times, the university had 1,282 professors in February 2012. Of those, 58 were women.[20] Comparing the number of professors in May 2020, there are 108 women among the 1,298 professors, which has almost doubled.[2] The university is steadily closing the gender gap, and by April 2021, half of its directors were women.[21]

In the fall of 2012 and for the first time, the University of Tokyo started two undergraduate programs entirely taught in English and geared toward international students—Programs in English at Komaba (PEAK)—the International Program on Japan in East Asia and the International Program on Environmental Sciences.[22][23] In 2014, the School of Science at the University of Tokyo introduced an all-English undergraduate transfer program called Global Science Course (GSC).[24]

Institute for Cosmic Ray Research, The University of Tokyo, started, on May 28, 2021, construction of the "Hyper-Kamiokande" device, for a new world-leading international scientific research project which is set to start experiments in 2027.[25]"""

# Tokenize the input article
inputs = tokenizer(ARTICLE, max_length=1024, truncation=True, return_tensors="pt")

# Generate summary
summary_ids = model.generate(inputs["input_ids"], num_beams=4, max_length=130, min_length=30, early_stopping=True)
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary_text)

### 4.7 機械翻訳

与えられた文を様々な言語に翻訳します。
言語の組み合わせに応じたモデルを読み込む必要があります。
あらゆる言語の組み合わせの翻訳が可能なわけではなく、どこかの研究グループがその組み合わせの学習済みモデルを提供してくれていれば、それを使えば可能ということです。
以下のコードは、ヘルシンキ大学が提供している機械翻訳モデルを使って日本語を英語に翻訳します。

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load tokenizer and model for Japanese-to-English translation
# 日本語から英語への翻訳のためのトークナイザーとモデルをロード
tokenizer_ja_en = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ja-en")
model_ja_en = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-ja-en")

japanese_text = "東京大学はいろいろな学問が学べて楽しい。"

# Tokenize the input Japanese text
# 入力された日本語テキストをトークン化
input_ids = tokenizer_ja_en.encode(japanese_text, return_tensors="pt")

# Generate the English translation
# 英語への翻訳を生成
translated_ids = model_ja_en.generate(input_ids)
english_text = tokenizer_ja_en.decode(translated_ids[0], skip_special_tokens=True)

print(english_text)

モデルを変えれば英語をドイツ語に翻訳することもできます。
英語は様々な言語に翻訳するモデルが提供されているので、日本語のようなマイナー言語間の機械翻訳は、いったん英語に翻訳したのち、それをさらに別の言語に翻訳するといった処理が行われることが多いです。

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Print the already translated Japanese to English text
# 日本語から英語への翻訳はMRC6Rdt_aCE3セルで実行済みで、結果はenglish_text変数に格納されています。
print('英語:', english_text)

# Load tokenizer and model for English-to-German and English-to-French translation using t5-base
# t5-baseモデルを使用して英語からドイツ語、および英語からフランス語への翻訳のためのトークナイザーとモデルをロード
tokenizer_t5 = AutoTokenizer.from_pretrained('t5-base')
model_t5 = AutoModelForSeq2SeqLM.from_pretrained('t5-base')

# Translate English to German
# 英語をドイツ語に翻訳
input_text_de = "translate English to German: " + english_text
input_ids_de = tokenizer_t5.encode(input_text_de, return_tensors="pt")
translated_ids_de = model_t5.generate(input_ids_de, max_length=512)
german_text = tokenizer_t5.decode(translated_ids_de[0], skip_special_tokens=True)
print('ドイツ語:', german_text)

# Translate English to French
# 英語をフランス語に翻訳
input_text_fr = "translate English to French: " + english_text
input_ids_fr = tokenizer_t5.encode(input_text_fr, return_tensors="pt")
translated_ids_fr = model_t5.generate(input_ids_fr, max_length=512)
french_text = tokenizer_t5.decode(translated_ids_fr[0], skip_special_tokens=True)
print('フランス語:', french_text)